# UNEP-CCC sectoral mitigation potentials 2035 — extract & clean

Source: *Bridging the gap: Sectoral greenhouse gas mitigation potentials in
2035* (UNEP-CCC & Common Futures, 2024), background report for EGR 2024
Chapter 6. `sample/sectoral-greenhouse-gas-emissions-reductions-potentials-in-2035.pdf`.

No machine-readable annex is published (verified 2026-06-05), so the
measure-level tables were **hand-transcribed** into §0 of this
notebook, which verifies every transcribed number against the PDF text
(pdfplumber) before writing the clean CSVs (all under `sample/`, which is **gitignored** — the source license is not an open license, so neither the PDF nor the transcribed tables are committed; see 'Getting the data' in review.md):

- `sample/unep2035_potentials_clean.csv` — Tables 3, 7–11, 13, 14 (measure level)
- `sample/unep2035_sector_summary.csv` — Table 1 (cross-assessment summary,
  overlap corrections, totals, emissions gap)

Verification strategy: a transcription error must show up as a number that
does not appear on the claimed PDF page, so every scalar is string-matched
against its source page's extracted text (plus aggregation sanity checks).
This catches transcription typos; it cannot catch a wrong-but-present number,
so a human spot-check of a few rows per table is still part of the review.

In [1]:
import hashlib, itertools, re
import pdfplumber
import pandas as pd

PDF = 'sample/sectoral-greenhouse-gas-emissions-reductions-potentials-in-2035.pdf'
EXPECTED_SHA256 = '7f804f7db2e50c28ee7d01ab3f4c471741dffe795222ebfa9f1c73efeb079fc1'

sha = hashlib.sha256(open(PDF, 'rb').read()).hexdigest()
assert sha == EXPECTED_SHA256, f'PDF changed: {sha}'
pdf = pdfplumber.open(PDF)
print('pages:', len(pdf.pages), '| sha256 ok')

pages: 53 | sha256 ok


## 0. Transcribed tables

Transcribed tables from UNEP-CCC 'Bridging the gap: Sectoral greenhouse gas
mitigation potentials in 2035' (2024). Hand-transcribed from the report PDF (pdfplumber text extraction is the verification oracle, see §3 below).

Values are recorded exactly as printed in the source tables, including two
cells we believe are typos (flagged, never silently corrected):
  - Table 3, Nuclear Energy 2030 = 5.9 (physically impossible vs the 10.3
    electricity aggregate; likely 0.59)
  - Table 1, Agriculture 2035 range (1.3-2.1) (conflicts with Table 8's
    2.01 (1.7-4.7); likely a copy of the methane row's range)

Schema (potentials csv): one row per measure/aggregate/baseline.
  sector        EGR sector grouping
  measure       measure or aggregate name as printed
  row_type      baseline | aggregate | measure | grand_total
  component     total | direct | indirect  (buildings only; total elsewhere)
  pot_2030/2035/2040           central value, GtCO2e/yr vs current-policy baseline
  unc_*_low / unc_*_high       uncertainty range as printed; missing = not stated
  source_table  table number in the report PDF
  source_refs   literature sources column as printed
  flags         semicolon-separated data-quality flags
  notes         anything a downstream user must know to not misuse the row

In [2]:
# row = (sector, measure, row_type, component,
#        p30, lo30, hi30, p35, lo35, hi35, p40, lo40, hi40,
#        source_table, source_refs, flags, notes)
N = None

POTENTIALS = [
    # ---- Table 3: energy (electricity + fossil fuel production), p.17 (pdf p.23)
    ("Energy", "Baseline emissions", "baseline", "total",
     18.7, N, N, 19.4, N, N, N, N, N, "Table 3", "UNEP (2024) Chapter 4", "",
     "Current-policy baseline for the energy sector."),
    ("Energy", "Energy sector (aggregated)", "aggregate", "total",
     12.2, N, N, 14.7, 13.2, 16.1, N, N, N, "Table 3", "", "",
     "Overlap-corrected; do not sum the measures below."),
    ("Energy", "Electricity sector (aggregated)", "aggregate", "total",
     10.3, 7.4, 11.8, 13.0, 11.9, 14.1, 15.0, 14.8, 15.2, "Table 3; Table 4", "", "",
     "2040 values from Table 4. 2030 fractions follow IPCC (2022b); "
     "2035 min = IEA NZE trajectory, max = assumed 95% reduction cap."),
    ("Energy", "Solar Energy", "measure", "total",
     4.2, N, N, 7.9, 7.0, 9.6, N, N, N, "Table 3",
     "IEA (2023a), DNV (2023), IRENA (2023), Nijsse (2023), Bogdanov (2019)", "",
     "High-electrification study potentials (17-22 TW) excluded as outliers."),
    ("Energy", "Wind Energy", "measure", "total",
     4.2, N, N, 7.7, 5.7, 8.9, N, N, N, "Table 3",
     "IEA (2023a), DNV (2023), IRENA (2023), Teske (2019)", "",
     "High-electrification study potentials (10-13 TW) excluded as outliers."),
    ("Energy", "Hydropower", "measure", "total",
     0.5, N, N, 1.0, 0.8, 1.2, N, N, N, "Table 3",
     "IEA (2023a), DNV (2023), IRENA (2023)", "", ""),
    ("Energy", "Nuclear Energy", "measure", "total",
     5.9, N, N, 0.8, 0.6, 1.0, N, N, N, "Table 3",
     "IEA (2023a), DNV (2023), IRENA (2023), NEA (2022)",
     "suspect_value_2030",
     "2030 value printed as 5.9 in the source PDF - physically impossible "
     "(electricity aggregate is 10.3, solar+wind alone are 8.4); likely a "
     "typo for 0.59. Recorded as printed; do not use the 2030 cell."),
    ("Energy", "Bioenergy excl. BECCS", "measure", "total",
     0.3, N, N, 0.5, 0.3, 0.7, N, N, N, "Table 3", "IEA (2023a), IRENA (2023)", "", ""),
    ("Energy", "Bioelectricity with CCS (BECCS)", "measure", "total",
     0.1, N, N, 0.5, 0.4, 0.6, N, N, N, "Table 3", "IEA (2023a)", "",
     "Body text warns dispatchable (BE)CCS can exceed 600 EUR/tCO2 at low "
     "full-load hours."),
    ("Energy", "Carbon Capture and Storage (CCS) excl. BECCS", "measure", "total",
     0.2, N, N, 0.5, 0.3, 0.6, N, N, N, "Table 3", "IEA (2023a)", "",
     "Body text gives more precise 0.17 (2030), 0.53 (2035), 0.64 (2040)."),
    ("Energy", "Geothermal", "measure", "total",
     0.5, N, N, 0.6, 0.2, 1.0, N, N, N, "Table 3",
     "IEA (2023a), IRENA (2023), Teske (2019)", "inconsistent_with_text_2030",
     "Body text gives 0.45 (0.1-0.8) for 2030 and 0.63 (0.2-1.0) for 2035; "
     "table prints rounded 0.5 / 0.6."),
    ("Energy", "Fossil fuel production (aggregated)", "aggregate", "total",
     1.9, 1.4, 2.4, 1.7, 1.3, 2.1, N, N, N, "Table 3", "", "", ""),
    ("Energy", "Reduce CH4 emissions from coal mining", "measure", "total",
     0.5, N, N, 0.4, 0.3, 0.5, 0.39, N, N, "Table 3; Table 7", "IEA (2024b)", "",
     "Table 7 gives more precise 0.49 (2030), 0.43 (2035), 0.39 (2040). "
     "Mostly negative-to-low cost."),
    ("Energy", "Reduce CH4 emissions from oil and gas", "measure", "total",
     1.4, N, N, 1.2, 0.9, 1.6, 1.11, N, N, "Table 3; Table 7", "IEA (2024b)", "",
     "Table 7 gives more precise 1.42 (2030), 1.24 (2035), 1.11 (2040). "
     "Mostly negative-to-low cost."),

    # ---- Table 8: AFOLU, p.28 (pdf p.34). Median + full range.
    ("AFOLU", "Baseline emissions", "baseline", "total",
     9.6, N, N, 9.5, N, N, N, N, N, "Table 8", "EPA (2019)", "",
     "Non-CO2 baseline from EPA (2019), 2035 linearly interpolated."),
    ("AFOLU", "AFOLU aggregated (Agriculture, Forests, Demand-side)", "aggregate", "total",
     8.02, 4.1, 16.7, 12.76, 6.3, 19.1, N, N, N, "Table 8", "", "",
     "Values are medians; ranges are FULL ranges, not central estimates. "
     "Overlap-corrected; do not sum measures."),
    ("AFOLU", "Agriculture (aggregated)", "aggregate", "total",
     1.42, 1.1, 3.9, 2.01, 1.7, 4.7, N, N, N, "Table 8", "", "",
     "Table 1 prints the 2035 range as (1.3-2.1) - inconsistent with this "
     "table; this table is taken as authoritative."),
    ("AFOLU", "Improved rice production", "measure", "total",
     0.2, 0.15, 0.22, 0.2, 0.15, 0.22, N, N, N, "Table 8",
     "Adapted from Beach et al. (2015); EPA (2019)", "", ""),
    ("AFOLU", "Nutrient management", "measure", "total",
     0.04, 0.03, 0.05, 0.04, 0.03, 0.05, N, N, N, "Table 8",
     "Adapted from Beach et al. (2015); EPA (2019)", "", ""),
    ("AFOLU", "Enteric fermentation", "measure", "total",
     0.17, 0.13, 0.18, 0.17, 0.08, 0.18, N, N, N, "Table 8",
     "Adapted from Beach et al. (2015); EPA (2019)", "", ""),
    ("AFOLU", "Manure management", "measure", "total",
     0.11, 0.07, 0.12, 0.1, 0.07, 0.12, N, N, N, "Table 8",
     "Adapted from Beach et al. (2015); EPA (2019)", "", ""),
    ("AFOLU", "Soil carbon management", "measure", "total",
     0.9, 0.4, 1.6, 1.5, 0.5, 2.4, N, N, N, "Table 8",
     "Adapted from IPCC AR6 WG3 Ch 7", "", ""),
    ("AFOLU", "Agroforestry on croplands and grasslands", "measure", "total",
     0.54, 0.4, 1.8, 0.54, 0.3, 1.8, N, N, N, "Table 8", "Naturebase", "", ""),
    ("AFOLU", "Biochar", "measure", "total",
     0.8, 0.3, 1.8, 1.1, 0.3, 1.8, N, N, N, "Table 8",
     "Adapted from IPCC AR6 WG3 Ch 7", "", ""),
    ("AFOLU", "Forests (aggregated)", "aggregate", "total",
     5.9, 2.7, 8.9, 8.35, 3.5, 11.7, N, N, N, "Table 8", "", "",
     "Excludes coastal wetlands, peatlands, grasslands (no updated economic "
     "data). >50% of forestry potential available under USD 50/tCO2e."),
    ("AFOLU", "Reduced deforestation", "measure", "total",
     1.8, 1.6, 4.0, 2.55, 1.8, 5.0, N, N, N, "Table 8",
     "Adapted from Austin et al. (2020)", "",
     "Lower than older assessments because current policies already cut "
     "deforestation in the baseline."),
    ("AFOLU", "Afforestation/ Reforestation", "measure", "total",
     2.6, 0.5, 3.0, 3.6, 0.9, 4.0, N, N, N, "Table 8",
     "Adapted from Austin et al. (2020)", "", ""),
    ("AFOLU", "Improved forest management", "measure", "total",
     1.5, 0.6, 1.9, 2.2, 0.8, 2.7, N, N, N, "Table 8",
     "Adapted from Austin et al. (2020)", "", ""),
    ("AFOLU", "Demand-side (aggregated)", "aggregate", "total",
     0.7, 0.4, 3.9, 2.4, 1.1, 3.7, N, N, N, "Table 8", "", "",
     "Diverted food production only; excludes land-use impacts (which would "
     "triple the potential)."),
    ("AFOLU", "Reduced food waste", "measure", "total",
     0.2, 0.1, 0.6, 0.7, 0.1, 1.0, N, N, N, "Table 8",
     "Adapted from IPCC AR6 WG3 Ch 7", "", ""),
    ("AFOLU", "Shift to sustainable healthy diets", "measure", "total",
     0.5, 0.3, 3.0, 1.7, 1.0, 2.7, N, N, N, "Table 8",
     "Adapted from IPCC AR6 WG3 Ch 7", "", ""),

    # ---- Table 9: buildings, p.29 (pdf p.35). Direct / indirect / totals.
    # Indirect = electricity use, attributed to the ENERGY sector in the
    # cross-sector aggregation (Table 1 overlap correction).
] + [
    ("Buildings", m, rt, comp, v30, N, N, v35, N, N, v40, N, N, "Table 9",
     "IPCC AR6 interpolated 2030-2040", fl, nt)
    for (m, rt, d, i, t, fl, nt) in [
        ("Avoid demand for energy services", "measure",
         (0.2, 0.3, 0.3), (0.3, 0.5, 0.6), (0.6, 0.8, 1.0), "",
         "Direct+indirect != total in print (rounding in source)."),
        ("New buildings - Better insulation", "measure",
         (0.2, 0.3, 0.3), (0.4, 0.6, 0.7), (0.6, 0.9, 1.1), "", ""),
        ("New buildings - Efficient generations of heat and cold", "measure",
         (0.2, 0.3, 0.3), (0.4, 0.4, 0.6), (0.6, 0.7, 0.8), "", ""),
        ("New buildings - Renewables", "measure",
         (0.0, 0.0, 0.0), (0.4, 0.6, 0.8), (0.4, 0.6, 0.8), "", ""),
        ("Retrofitting - Better insulation", "measure",
         (0.1, 0.1, 0.2), (0.0, 0.0, 0.0), (0.2, 0.2, 0.2), "", ""),
        ("Retrofitting - Efficient generation of heat and cold", "measure",
         (0.1, 0.1, 0.1), (0.0, 0.1, 0.1), (0.1, 0.1, 0.1), "",
         "Direct+indirect != total in print (rounding in source)."),
        ("Appliances", "measure",
         (0.2, 0.2, 0.3), (0.5, 0.7, 0.9), (0.7, 0.9, 1.1), "",
         "IEA (2024e) suggests potential could be ~double; excluded due to "
         "unclear baselines."),
        ("Buildings (aggregated)", "aggregate",
         (1.1, 1.2, 1.4), (2.2, 2.9, 3.7), (3.2, 4.2, 5.2), "",
         "Table 1 gives 2035 uncertainty (3.1-5.2) for the total."),
    ]
    for comp, (v30, v35, v40) in [("direct", d), ("indirect", i), ("total", t)]
] + [
    # ---- Table 10: transport, p.32 (pdf p.38)
    ("Transport", "Baseline emissions", "baseline", "total",
     8.8, N, N, 9.0, N, N, 9.0, N, N, "Table 10", "UNEP (2024) Chapter 4", "",
     "Baseline is 0.5 (2030) / 0.9 (2035) GtCO2e HIGHER than the IEA STEPS "
     "numbers the potentials are largely based on, so potentials may be "
     "conservative (source footnote)."),
    ("Transport", "Transport (aggregated)", "aggregate", "total",
     3.2, 1.6, 4.8, 4.8, 2.4, 7.2, 6.1, 3.0, 9.1, "Table 10", "", "", ""),
    ("Transport", "Road transport (aggregated)", "aggregate", "total",
     2.5, 1.2, 3.7, 3.6, 1.8, 5.4, 4.4, 2.2, 6.6, "Table 10", "", "", ""),
    ("Transport", "Shifts to public transport", "measure", "total",
     0.8, 0.4, 1.2, 1.1, 0.5, 1.6, 1.4, 0.7, 2.1, "Table 10",
     "ITDP & UC Davis (2021, 2015)", "", ""),
    ("Transport", "Shifts to bikes and e-bikes", "measure", "total",
     0.3, 0.1, 0.4, 0.3, 0.2, 0.5, 0.4, 0.2, 0.5, "Table 10",
     "ITDP & UC Davis (2021, 2015)", "", ""),
    ("Transport", "Shift to electric LDV", "measure", "total",
     0.3, 0.2, 0.5, 0.6, 0.3, 1.0, 3.7, 1.9, 5.6, "Table 10",
     "IEA (2024c) Global EV Outlook", "",
     "Low vs IPCC AR6 (0.8 in 2030) because recent EV growth moved into the "
     "baseline; 2040 jump reflects fleet turnover."),
    ("Transport", "Shift to electric HDV", "measure", "total",
     0.1, 0.0, 0.1, 0.2, 0.1, 0.3, N, N, N, "Table 10",
     "IEA (2024c) Global EV Outlook", "", ""),
    ("Transport", "Fuel efficiency LDV", "measure", "total",
     0.5, 0.3, 0.8, 0.7, 0.4, 1.1, N, N, N, "Table 10",
     "Based on IEA (2023a) NZE", "", ""),
    ("Transport", "Fuel efficiency HDV", "measure", "total",
     0.6, 0.3, 0.9, 1.1, 0.5, 1.6, N, N, N, "Table 10",
     "Based on IEA (2023a) NZE", "", ""),
    ("Transport", "Biofuels", "measure", "total",
     0.2, 0.1, 0.3, 0.2, 0.1, 0.3, 0.2, 0.1, 0.2, "Table 10",
     "IEA (2023a) NZE", "", ""),
    ("Transport", "Shipping (aggregated)", "aggregate", "total",
     0.2, 0.1, 0.3, 0.4, 0.2, 0.6, 0.6, 0.3, 0.9, "Table 10", "", "", ""),
    ("Transport", "Shipping: energy efficiency, optimisation, low/zero-emission fuels",
     "measure", "total",
     0.2, 0.1, 0.3, 0.4, 0.2, 0.6, 0.6, 0.3, 0.9, "Table 10",
     "IEA (2023a) NZE", "", "Single measure; identical to shipping aggregate."),
    ("Transport", "Aviation (aggregated)", "aggregate", "total",
     0.5, 0.3, 0.8, 0.8, 0.4, 1.2, 1.1, 0.5, 1.6, "Table 10", "", "", ""),
    ("Transport", "Aviation: reduced demand increase", "measure", "total",
     0.4, 0.2, 0.6, 0.5, 0.3, 0.8, 0.7, 0.3, 1.0, "Table 10",
     "Bergero et al. (2023)", "",
     "Assumes 1%/yr demand growth vs 4% BAU - a substantial deviation from "
     "the historical demand-income correlation."),
    ("Transport", "Aviation: energy efficiency and optimisation", "measure", "total",
     0.1, 0.0, 0.1, 0.1, 0.1, 0.2, 0.2, 0.1, 0.3, "Table 10",
     "Based on ICAO (2022)", "", ""),
    ("Transport", "Aviation: shift to low- and zero-emission fuels", "measure", "total",
     0.1, 0.0, 0.1, 0.2, 0.1, 0.3, 0.4, 0.2, 0.6, "Table 10",
     "Based on ICAO (2022)", "", ""),
    ("Transport", "Aviation: other", "measure", "total",
     0.0, 0.0, 0.0, 0.1, 0.0, 0.1, 0.1, 0.0, 0.1, "Table 10",
     "Based on ICAO (2022)", "", ""),

    # ---- Table 11: industry, p.36 (pdf p.42)
    ("Industry", "Baseline emissions", "baseline", "total",
     14.2, 13.4, 15.1, 14.7, 13.1, 16.4, 15.2, 12.9, 17.8, "Table 11", "", "",
     "Includes all onsite process CO2 (e.g. coking, BFBOF top gases) - a "
     "different boundary than EGR Chapter 2."),
    ("Industry", "Industry (aggregated)", "aggregate", "total",
     5.2, 4.9, 5.6, 7.7, 6.9, 8.8, 10.2, 7.6, 12.7, "Table 11", "", "",
     "UNCORRECTED for autonomous implementation - use the corrected row for "
     "cross-sector comparisons (Table 1 uses the corrected one)."),
    ("Industry", "Industry (aggregated), corrected for autonomous implementation",
     "aggregate", "total",
     4.4, 4.2, 4.8, 6.6, 5.8, 7.4, 8.7, 6.5, 10.8, "Table 11", "", "",
     "15% of total potential assumed to happen autonomously (market-driven)."),
    ("Industry", "Energy efficiency", "measure", "total",
     1.0, 1.0, 1.1, 1.1, 1.0, 1.2, 1.1, 1.0, 1.3, "Table 11", "", "",
     "Partially market-driven (see autonomous-implementation correction)."),
    ("Industry", "Material efficiency", "measure", "total",
     0.7, 0.7, 0.8, 1.2, 1.1, 1.4, 1.8, 1.5, 2.1, "Table 11", "", "", ""),
    ("Industry", "Enhanced recycling", "measure", "total",
     0.6, 0.5, 0.6, 1.0, 0.9, 1.1, 1.3, 1.2, 1.6, "Table 11", "", "", ""),
    ("Industry", "Fuel switching and electrification", "measure", "total",
     1.6, 1.5, 1.7, 2.1, 1.8, 2.3, 2.6, 2.2, 3.1, "Table 11", "", "",
     "'Viable but uncompetitive without climate policies' (source)."),
    ("Industry", "Advanced feedstock decarbonization & process changes", "measure", "total",
     0.7, 0.7, 0.8, 1.2, 1.1, 1.3, 1.7, 1.4, 2.0, "Table 11", "", "", ""),
    ("Industry", "CCU and CCS", "measure", "total",
     0.1, 0.1, 0.1, 0.5, 0.4, 0.6, 0.8, 0.7, 0.9, "Table 11", "", "", ""),
    ("Industry", "Cementitious material substitution", "measure", "total",
     0.3, 0.3, 0.3, 0.4, 0.4, 0.5, 0.6, 0.5, 0.7, "Table 11", "", "",
     "E.g. 1/3 ground limestone & 2/3 calcined clays, replacing <=50% clinker."),
    ("Industry", "Reduction of N2O emissions", "measure", "total",
     0.2, 0.2, 0.2, 0.3, 0.3, 0.3, 0.3, 0.3, 0.4, "Table 11", "", "", ""),

    # ---- Table 13: waste, p.41 (pdf p.47). Mid-range; 2035 full range.
    ("Waste", "Solid waste (CH4)", "measure", "total",
     0.63, N, N, 0.75, 0.65, 0.84, 0.89, N, N, "Table 13",
     "EPA (2019); Hoglund-Isaksson et al. (2020); IPCC (2022b)", "",
     "GWP for biogenic methane 27.2. High end = integral waste management; "
     "low end = landfill gas recovery."),
    ("Waste", "Wastewater (CH4)", "measure", "total",
     0.20, N, N, 0.29, 0.28, 0.31, 0.34, N, N, "Table 13",
     "EPA (2019); Hoglund-Isaksson et al. (2020); IPCC (2022b)", "", ""),
    ("Waste", "Waste (aggregated)", "aggregate", "total",
     0.83, N, N, 1.04, 0.95, 1.21, 1.23, N, N, "Table 13", "", "",
     "2035 mid-range is the average of available estimates."),

    # ---- Table 14: fluorinated gases, p.41 (pdf p.47). Ranges only.
    ("F-gases", "Fluorinated gases", "measure", "total",
     N, 0.80, 1.42, N, 1.01, 1.66, N, 1.30, 2.03, "Table 14",
     "EPA (2019, 2024); Purohit & Hoglund-Isaksson (2017)", "range_only",
     "No central estimate given; Table 1 uses ~midpoints (1.2 in 2030, 1.4 "
     "in 2035). Kigali Amendment already regulates much of this."),

    # ---- Grand total (Table 1 / executive summary)
    ("All sectors", "Total mitigation potential (corrected for overlap)",
     "grand_total", "total",
     31.0, 25.0, 35.0, 41.0, 36.0, 46.0, N, N, N, "Table 1", "", "",
     "Cross-sector total after overlap corrections (-2.3 in 2030, -3.9 in "
     "2035, mostly electricity x buildings/industry). Compare to emissions "
     "gap of 24 (20-26) in 2030 and 32 (20-37) in 2035."),
]

# ---- Table 1 (p.7, pdf p.13): cross-assessment sector summary.
# columns: row label, unep2017_2030, ipcc_ar6_2030, egr2024_2030,
#          egr2024_2035, range35_lo, range35_hi, flags, notes
SECTOR_SUMMARY = [
    ("Electricity production", 10.3, 11.0, 10.3, 13.0, 11.9, 14.1, "", ""),
    ("Methane from fossil fuels", 2.2, 1.6, 1.9, 1.7, 1.3, 2.1, "", ""),
    ("Agriculture", 6.7, 4.1, 1.4, 2.0, 1.3, 2.1, "suspect_range_2035",
     "2035 range printed as (1.3-2.1), identical to the methane row and "
     "inconsistent with Table 8's (1.7-4.7); likely a copy error. UNEP 2017 "
     "column included demand-side."),
    ("Forestry", 5.3, 7.3, 5.9, 8.4, 3.5, 11.7, "", ""),
    ("AFOLU demand side", None, 2.2, 0.7, 2.4, 1.1, 3.7, "",
     "UNEP 2017: included in agriculture."),
    ("Buildings", 5.9, 3.2, 3.2, 4.2, 3.1, 5.2, "",
     "Direct + indirect. Indirect overlaps electricity (see corrections)."),
    ("Transport", 4.7, 3.8, 3.2, 4.8, 2.4, 7.2, "", ""),
    ("Industry", 5.4, 5.4, 4.4, 6.6, 5.8, 7.4, "",
     "Corrected for autonomous implementation."),
    ("Fluorinated gases", None, 1.2, 1.2, 1.4, 1.0, 1.8, "",
     "Table 14 gives ranges only; these are ~midpoints."),
    ("Waste and wastewater", 0.4, 0.7, 0.8, 1.0, 0.9, 1.2, "", ""),
    ("Correction for overlap between sectors", None, -1.0, -2.3, -3.9, None, None, "",
     "EGR 2024: electricity x buildings -1.6 (2030) / -2.9 (2035); "
     "electricity x industry -0.7 / -1.0. UNEP 2017: included in sector estimates."),
    ("Total (corrected for overlap)", 38.0, 38.0, 31.0, 41.0, 36.0, 46.0, "",
     "UNEP 2017 range (35-41); IPCC AR6 range (32-44); EGR 2030 range (25-35). "
     "Cost cut-off: $100/tCO2e for UNEP 2017 and IPCC AR6 columns, $200 for EGR 2024."),
    ("Emissions gap for achieving 1.5C", None, None, 24.0, 32.0, 20.0, 37.0, "",
     "UNEP (2024) Chapter 4; 2030 range (20-26)."),
]

HEADER_POTENTIALS = [
    "sector", "measure", "row_type", "component",
    "pot_2030", "unc_2030_low", "unc_2030_high",
    "pot_2035", "unc_2035_low", "unc_2035_high",
    "pot_2040", "unc_2040_low", "unc_2040_high",
    "source_table", "source_refs", "flags", "notes",
]
HEADER_SUMMARY = [
    "row", "unep2017_2030", "ipcc_ar6_2030", "egr2024_2030", "egr2024_2035",
    "egr2024_2035_unc_low", "egr2024_2035_unc_high", "flags", "notes",
]

## 1. Locate source tables

Pages (1-based, PDF order) for each table used:

In [3]:
TABLE_PAGES = {        # pdf page (1-based) holding the table body
    'Table 1': [13],
    'Table 3': [23],
    'Table 4': [24],
    'Table 7': [32],
    'Table 8': [34],
    'Table 9': [35],
    'Table 10': [38],
    'Table 11': [42],
    'Table 13': [47],
    'Table 14': [47],
}
PAGE_TEXT = {n: (pdf.pages[n-1].extract_text() or '') for n in
             set(itertools.chain.from_iterable(TABLE_PAGES.values()))}
for t, pages in TABLE_PAGES.items():
    for n in pages:
        assert f'{t}.' in PAGE_TEXT[n], (t, n)
print('all table captions found on expected pages')

all table captions found on expected pages


## 2. Build dataframes from the transcription

In [4]:
pot = pd.DataFrame(POTENTIALS, columns=HEADER_POTENTIALS)
summ = pd.DataFrame(SECTOR_SUMMARY, columns=HEADER_SUMMARY)
pot.insert(13, 'unit', 'GtCO2e/yr')
print(pot.groupby(['sector', 'row_type']).size().unstack(fill_value=0))
pot.head(3)

row_type     aggregate  baseline  grand_total  measure
sector                                                
AFOLU                4         1            0       12
All sectors          0         0            1        0
Buildings            3         0            0       21
Energy               3         1            0       10
F-gases              0         0            0        1
Industry             2         1            0        8
Transport            4         1            0       12
Waste                1         0            0        2


,sector,measure,row_type,component,pot_2030,unc_2030_low,unc_2030_high,pot_2035,unc_2035_low,unc_2035_high,pot_2040,unc_2040_low,unc_2040_high,unit,source_table,source_refs,flags,notes
0,Energy,Baseline emissions,baseline,total,18.7,NaN,NaN,19.4,NaN,NaN,NaN,NaN,NaN,GtCO2e/yr,Table 3,UNEP (2024) Chapter 4,,Current-policy baseline for the energy sector.
1,Energy,Energy sector (aggregated),aggregate,total,12.2,NaN,NaN,14.7,13.2,16.1,NaN,NaN,NaN,GtCO2e/yr,Table 3,,,Overlap-corrected; do not sum the measures below.
2,Energy,Electricity sector (aggregated),aggregate,total,10.3,7.4,11.8,13.0,11.9,14.1,15.0,14.8,15.2,GtCO2e/yr,Table 3; Table 4,,,2040 values from Table 4. 2030 fractions follo...


## 3. Verify every transcribed scalar appears on its source page

Numbers are matched as printed strings (e.g. `12.76`, `0.04`); a value is
also accepted if it appears without a trailing zero or with the page's
de-hyphenation quirks. Failures are listed; the run asserts none.

In [5]:
def norm(s):
    s = s.replace('\u2013', '-').replace('\u2212', '-')
    return re.sub(r'\s+', ' ', s)

def variants(x):
    if x is None or x != x:
        return []
    v = {f'{x:g}'}
    if isinstance(x, float) and x == int(x):
        v |= {f'{int(x)}', f'{int(x)}.0'}
    if isinstance(x, float):
        v.add(f'{x:.2f}'.rstrip('0').rstrip('.'))
        v.add(f'{x:.1f}')
        v.add(f'{x:.2f}')
    return v

def on_pages(x, tables):
    pages = set(itertools.chain.from_iterable(
        TABLE_PAGES[t.strip()] for t in tables.split(';')))
    text = norm(' '.join(PAGE_TEXT[n] for n in pages))
    return any(s in text for s in variants(x))

failures = []
numcols = [c for c in pot.columns if c.startswith(('pot_', 'unc_'))]
for _, r in pot.iterrows():
    for c in numcols:
        if pd.notna(r[c]) and not on_pages(abs(r[c]), r['source_table']):
            failures.append((r['measure'], c, r[c]))

p13 = norm(PAGE_TEXT[13])
for _, r in summ.iterrows():
    for c in summ.columns[1:7]:
        if pd.notna(r[c]) and not any(s in p13 for s in variants(abs(r[c]))):
            failures.append(('Table 1: ' + r['row'], c, r[c]))

for f in failures:
    print('MISSING ON PAGE:', f)
assert not failures, f'{len(failures)} transcribed values not found in PDF text'
print(f'all {sum(pot[c].notna().sum() for c in numcols) + int(summ[summ.columns[1:7]].notna().sum().sum())} scalars verified against PDF text')

all 568 scalars verified against PDF text


## 4. Internal consistency checks

The source warns measures cannot be summed (overlaps), so these are
*sanity* bounds, not equalities. Known source quirks are asserted
explicitly so they stay visible.

In [6]:
def row(measure):
    return pot[(pot.measure == measure) & (pot.component == 'total')].iloc[0]

# Aggregates must not exceed their baselines
for sec, agg, base in [
    ('Energy', 'Energy sector (aggregated)', 'Baseline emissions'),
    ('Transport', 'Transport (aggregated)', 'Baseline emissions'),
]:
    a = pot[(pot.sector == sec) & (pot.measure == agg)].iloc[0]
    b = pot[(pot.sector == sec) & (pot.measure == base)].iloc[0]
    assert a.pot_2035 <= b.pot_2035, sec

# Known typo: nuclear 2030 as printed exceeds the electricity aggregate
assert row('Nuclear Energy').pot_2030 == 5.9
assert row('Nuclear Energy').pot_2030 > row('Electricity sector (aggregated)').pot_2030 * 0.5, \
    'flagged-as-typo cell no longer looks anomalous - re-check source'
assert 'suspect_value_2030' in row('Nuclear Energy')['flags']

# Waste components sum to the aggregate
w = pot[pot.sector == 'Waste']
assert abs(w[w.row_type == 'measure'].pot_2035.sum() - row('Waste (aggregated)').pot_2035) < 0.011

# Table 1 grand total reconciles: sector rows + overlap correction ~ total (rounding +-1)
s30 = summ.iloc[0:10].egr2024_2030.sum() + summ.iloc[10].egr2024_2030
s35 = summ.iloc[0:10].egr2024_2035.sum() + summ.iloc[10].egr2024_2035
assert abs(s30 - 31) <= 1 and abs(s35 - 41) <= 1, (s30, s35)
print(f'consistency checks pass (Table 1 reconstructed totals: {s30:.1f} -> 31, {s35:.1f} -> 41)')

consistency checks pass (Table 1 reconstructed totals: 30.7 -> 31, 41.6 -> 41)


## 5. Write clean CSVs

In [7]:
pot.to_csv('sample/unep2035_potentials_clean.csv', index=False)
summ.to_csv('sample/unep2035_sector_summary.csv', index=False)
print(len(pot), 'rows ->', 'sample/unep2035_potentials_clean.csv')
print(len(summ), 'rows ->', 'sample/unep2035_sector_summary.csv')

88 rows -> sample/unep2035_potentials_clean.csv
13 rows -> sample/unep2035_sector_summary.csv


## 6. Findings recorded during extraction

- **Nuclear 2030 = 5.9 is a typo in the published PDF** (verified at
  character level in the PDF). Solar+wind alone are 8.4 against a 10.3
  electricity aggregate; the value is likely 0.59. Kept as printed,
  flagged `suspect_value_2030`.
- **Table 1's Agriculture 2035 range (1.3–2.1)** duplicates the methane
  row's range and contradicts Table 8's (1.7–4.7). Table 8 treated as
  authoritative; flagged `suspect_range_2035` in the summary CSV.
- **Geothermal**: table prints 0.5/0.6; body text gives 0.45 (0.1–0.8)
  and 0.63 (0.2–1.0). Flagged.
- **2030 measure-level uncertainty ranges are missing in Table 3**
  (energy) — only aggregates have them; Table 1 notes "similar
  uncertainty ranges apply" across years.
- **Buildings direct+indirect ≠ total** in two rows (source rounding).
- **F-gases have no central estimate** (Table 14, `range_only`).
- Waste uses **GWP 27.2 for biogenic methane** — mind GWP consistency
  when comparing with other datasets.
- The industry chapter references "supplementary material" that is not
  published with the report; Table 12 (assumptions) is in the PDF and
  was left untranscribed (parameters, not potentials).

## 7. Crosswalk and comparison with SPM.7

Builds `sample/spm7_unep_comparison.csv` — the 29-pair crosswalk to
`ipcc-ar6-spm7-mitigation-potentials` behind the review's comparison
section and the decision to keep SPM.7 as the ranking source. Where UNEP
is finer than an SPM.7 option, UNEP measures are summed (overlap risk
accepted: components of one option are assessed jointly upstream).
Nuclear 2030 is excluded (typo cell).

In [8]:
IPCC_CSV = ('../../../../ipcc/ipcc-ar6-spm7-mitigation-potentials/'
            'releases/2023/data/spm7a_mitigation_options_2030_clean.csv')
ipcc = pd.read_csv(IPCC_CSV).set_index('option')
uu = pot[pot.component == 'total'].set_index('measure')

XW = {
 'Solar': ['Solar Energy'],
 'Wind': ['Wind Energy'],
 'Reduce CH4 from coal, oil and gas': ['Fossil fuel production (aggregated)'],
 'Bioelectricity (includes BECCS)': ['Bioenergy excl. BECCS', 'Bioelectricity with CCS (BECCS)'],
 'Geothermal and hydropower': ['Geothermal', 'Hydropower'],
 'Nuclear': ['Nuclear Energy'],
 'Fossil Carbon Capture and Storage (CCS)': ['Carbon Capture and Storage (CCS) excl. BECCS'],
 'Reduce conversion of natural ecosystems': ['Reduced deforestation'],
 'Carbon sequestration in agriculture': ['Soil carbon management', 'Biochar',
                                         'Agroforestry on croplands and grasslands'],
 'Ecosystem restoration, afforestation, reforestation': ['Afforestation/ Reforestation'],
 'Shift to sustainable healthy diets': ['Shift to sustainable healthy diets'],
 'Forest and fire management': ['Improved forest management'],
 'Reduce CH4 and N2O in agriculture': ['Improved rice production', 'Nutrient management',
                                       'Enteric fermentation', 'Manure management'],
 'Reduce food loss and food waste': ['Reduced food waste'],
 'Fuel efficient vehicles': ['Fuel efficiency LDV', 'Fuel efficiency HDV'],
 'Electric vehicles': ['Shift to electric LDV', 'Shift to electric HDV'],
 'Efficient lighting, appliances and equipment': ['Appliances'],
 'Public transport and bicycling': ['Shifts to public transport', 'Shifts to bikes and e-bikes'],
 'Biofuels': ['Biofuels'],
 'Efficient shipping and aviation': ['Shipping (aggregated)',
                                     'Aviation: energy efficiency and optimisation',
                                     'Aviation: shift to low- and zero-emission fuels'],
 'Avoid demand for energy services': ['Avoid demand for energy services'],
 'Onsite renewables': ['New buildings - Renewables'],
 'Fuel switching': ['Fuel switching and electrification'],
 'Energy efficiency': ['Energy efficiency'],
 'Material efficiency': ['Material efficiency'],
 'Reduce CH4 from waste/wastewater': ['Waste (aggregated)'],
 'Construction materials substitution': ['Cementitious material substitution'],
 'Enhanced recycling': ['Enhanced recycling'],
 'Carbon capture with utilization and storage': ['CCU and CCS'],
}

rows = []
for opt, ms in XW.items():
    p30 = round(sum(uu.loc[m, 'pot_2030'] for m in ms), 2)
    p35 = round(sum(uu.loc[m, 'pot_2035'] for m in ms), 2)
    note = ''
    if opt == 'Nuclear':
        p30, note = None, 'UNEP 2030 cell is a typo (suspect_value_2030); excluded'
    if opt == 'Reduce conversion of natural ecosystems':
        note = 'scope differs: UNEP reduced deforestation excludes non-forest ecosystems'
    rows.append({'spm7_option': opt, 'unep_measures': '; '.join(ms),
                 'ipcc_pot_2030': ipcc.loc[opt, 'total'],
                 'ipcc_unc_low': ipcc.loc[opt, 'unc_low'],
                 'ipcc_unc_high': ipcc.loc[opt, 'unc_high'],
                 'unep_pot_2030': p30, 'unep_pot_2035': p35, 'notes': note})
cmp_df = pd.DataFrame(rows)
cmp_df.to_csv('sample/spm7_unep_comparison.csv', index=False)

from scipy.stats import spearmanr
d30 = cmp_df.dropna(subset=['unep_pot_2030'])
r30 = spearmanr(d30.ipcc_pot_2030, d30.unep_pot_2030).statistic
r35 = spearmanr(cmp_df.ipcc_pot_2030, cmp_df.unep_pot_2035).statistic
assert r30 > 0.75 and r35 > 0.8, 'ordinal agreement degraded - revisit the decision in review.md'
print(f'{len(cmp_df)} pairs -> sample/spm7_unep_comparison.csv | Spearman: 2030v2030 {r30:.2f}, 2030v2035 {r35:.2f}')

29 pairs -> sample/spm7_unep_comparison.csv | Spearman: 2030v2030 0.81, 2030v2035 0.88
